In [2]:
# Celda 7: Entrenamiento del Modelo Baseline (TF-IDF + Regresión Logística)
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix

# 1. Cargar el dataset lematizado y preprocesado del Día 3
df = pd.read_csv("dataset_tecnico_preprocesado.csv")

# Eliminar posibles valores nulos remanentes tras la limpieza
df = df.dropna(subset=['texto_limpio', 'categoria'])

# 2. Separar características (X) y variable objetivo / etiquetas (y)
X = df['texto_limpio']
y = df['categoria']

# 3. Partición de datos: 80% Entrenamiento y 20% Prueba (Set ciego de evaluación)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.20, 
    random_state=42, 
    stratify=y  # Mantiene la misma proporción de categorías en ambos sets
)

print(f"📊 Datos de entrenamiento: {X_train.shape[0]} documentos.")
print(f"📊 Datos de prueba (evaluación): {X_test.shape[0]} documentos.")

# 4. Construcción del Pipeline de Scikit-Learn
# Encapsulamos el Vectorizador y el Clasificador para asegurar que el procesamiento sea idéntico
modelo_baseline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1, 2))), 
    ('clf', LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000))
])

# 5. Entrenamiento del Modelo Baseline
print("\n⏳ Entrenando el modelo baseline (TF-IDF + Regresión Logística)...")
modelo_baseline.fit(X_train, y_train)
print("✅ ¡Entrenamiento completado!")

# 6. Evaluación del rendimiento en el set de prueba
y_pred = modelo_baseline.predict(X_test)

print("\n📋 === REPORTE DE CLASIFICACIÓN (MÉTRICA DEL HACKATHON) ===")
print(classification_report(y_test, y_pred))

📊 Datos de entrenamiento: 4000 documentos.
📊 Datos de prueba (evaluación): 1000 documentos.

⏳ Entrenando el modelo baseline (TF-IDF + Regresión Logística)...
✅ ¡Entrenamiento completado!

📋 === REPORTE DE CLASIFICACIÓN (MÉTRICA DEL HACKATHON) ===
              precision    recall  f1-score   support

     backend       0.69      0.76      0.72       250
       cloud       0.73      0.75      0.74       250
 datascience       0.90      0.80      0.84       250
    frontend       0.74      0.73      0.73       250

    accuracy                           0.76      1000
   macro avg       0.76      0.76      0.76      1000
weighted avg       0.76      0.76      0.76      1000



In [3]:
# Celda 8: Búsqueda en Rejilla (GridSearchCV) para Optimizar C y TF-IDF
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

# 1. Cargar el dataset lematizado
df = pd.read_csv("dataset_tecnico_preprocesado.csv")
df = df.dropna(subset=['texto_limpio', 'categoria'])

X = df['texto_limpio']
y = df['categoria']

# 2. Partición balanceada (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 3. Definir el Pipeline base
pipeline_base = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, min_df=2, max_df=0.85)),
    ('clf', LogisticRegression(class_weight='balanced', random_state=42, max_iter=1500))
])

# 4. Definir la rejilla de parámetros a evaluar
# Probaremos valores de C más altos (menos regularización) para afinar fronteras difusas
param_grid = {
    'tfidf__ngram_range': [(1, 2), (1, 3)],  # Evaluar si frases de 3 palabras aportan contexto técnico
    'clf__C': [1.0, 3.0, 5.0, 10.0]          # Fuerza de regularización inversa
}

# 5. Configurar la búsqueda cruzada (usando 3 folds para optimizar tiempo en el Hackathon)
print("⏳ Iniciando GridSearchCV buscando el mejor F1-Score Macro...")
grid_search = GridSearchCV(
    estimator=pipeline_base,
    param_grid=param_grid,
    scoring='f1_macro',  # Optimizamos directamente la métrica clave del Hackathon
    cv=3,
    verbose=1,
    n_jobs=1  # Usa todos los núcleos disponibles de tu procesador para acelerar el cálculo
)

# Ejecutar la búsqueda
grid_search.fit(X_train, y_train)

# 6. Resultados del tuning
print("\n🏆 ¡Búsqueda completada con éxito!")
print(f"Mejores parámetros encontrados: {grid_search.best_params_}")
print(f"Mejor F1-Score estimado en validación cruzada: {grid_search.best_score_:.4f}")

# 7. Evaluar el modelo optimizado final en el set de prueba ciego
mejor_modelo = grid_search.best_estimator_
y_pred = mejor_modelo.predict(X_test)

print("\n📋 === REPORTE DE CLASIFICACIÓN CON TUNING FINAL DE C ===")
print(classification_report(y_test, y_pred))

⏳ Iniciando GridSearchCV buscando el mejor F1-Score Macro...
Fitting 3 folds for each of 8 candidates, totalling 24 fits

🏆 ¡Búsqueda completada con éxito!
Mejores parámetros encontrados: {'clf__C': 3.0, 'tfidf__ngram_range': (1, 2)}
Mejor F1-Score estimado en validación cruzada: 0.7608

📋 === REPORTE DE CLASIFICACIÓN CON TUNING FINAL DE C ===
              precision    recall  f1-score   support

     backend       0.70      0.74      0.72       250
       cloud       0.75      0.75      0.75       250
 datascience       0.87      0.81      0.84       250
    frontend       0.73      0.74      0.73       250

    accuracy                           0.76      1000
   macro avg       0.76      0.76      0.76      1000
weighted avg       0.76      0.76      0.76      1000



In [4]:
# Celda 9: Exportación oficial del modelo certificado
import joblib

# Guardamos el mejor estimador encontrado por GridSearchCV
joblib.dump(grid_search.best_estimator_, "modelo_baseline_v2.joblib")
print("💾 ¡Artefacto 'modelo_baseline_v2.joblib' certificado y listo para producción!")

💾 ¡Artefacto 'modelo_baseline_v2.joblib' certificado y listo para producción!
